# Qwen3.5-27B QLoRA: extend adaptation to Gated DeltaNet

Compare the NF4 **Qwen/Qwen3.5-27B baseline** with **QLoRA (r=48, alpha=96)** over seeds **42, 123, and 2026**.

- **Independent variable:** extend the existing seven attention/MLP targets with Gated DeltaNet's `in_proj_qkv` and `out_proj`. Other DeltaNet projections and the vision encoder stay frozen.
- Preserve `DiffModel/Qwen3.5-27B.ipynb`'s dataset, question splits, prompt, completion masks, quantisation, training settings, checkpoint selection, decoding, and metrics.
- Train one fresh adapter per seed and select its lowest-validation-loss checkpoint. Evaluate **Baseline** (adapter disabled) and **QLoRA** (adapter enabled) on that seed's identical held-out rows: three adapters and six evaluations.
- Report per-seed metrics, equal-weight means, sample standard deviations, and QLoRA-minus-baseline differences. Across-seed variation includes training randomness and changing held-out questions.

Run all cells from the top. Results populate after all six evaluations finish and are saved in a separate experiment directory. To assess the effect of the extra targets, compare these QLoRA results with the original notebook's QLoRA results; the baseline here is the unadapted model.


## Install dependencies

- Use Python 3.12 in the Linux CUDA training environment, with the isolated **Python (qlora)** kernel and `PYTHONNOUSERSITE=1`. Run setup before imports, including after a kernel restart.
- Reuse Scaling's active-kernel installation, NumPy/analysis dependencies, CUDA compiler diagnostics, and pinned `torch==2.10.0+cu126` wheel matching the cluster's CUDA 12.6 toolkit. For another toolkit, set both `QLORA_TORCH_PACKAGE` and `QLORA_TORCH_INDEX_URL` before setup.
- Keep the source notebook's dependency setup: `transformers>=5.16.1`, `torchvision`, and `kernels` for the multimodal loader. Qwen uses SDPA for full attention and the PyTorch fallback for DeltaNet linear attention; no additional kernel packages are installed. The fallback can be slower and use more memory.
- Installation errors stop this cell. Restart the kernel after changing imported packages, then rerun setup. Package versions, checkpoint revision, dataset hash, and split membership are saved in `experiment.json`; reproduce an earlier run with its recorded versions.
- CUDA training requires a compatible NVIDIA driver and CUDA-enabled PyTorch. A separate matching CUDA compiler is needed only for compiled extensions; load the cluster toolchain before launching the notebook.
- References: [checkpoint configuration](https://huggingface.co/Qwen/Qwen3.5-27B/blob/main/config.json), [Qwen3.5 model documentation](https://huggingface.co/docs/transformers/model_doc/qwen3_5), [native chat template](https://huggingface.co/Qwen/Qwen3.5-27B/blob/main/chat_template.jinja), [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer).


In [1]:
import os
import re
import shutil
import site
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, distribution, version
from pathlib import Path

print("Python:", sys.executable)
print("Environment:", sys.prefix)
# Disable user-site packages at kernel startup; changing the variable here
# cannot remove packages already loaded by this kernel.
user_sites = site.getusersitepackages()
if isinstance(user_sites, str):
    user_sites = [user_sites]
if any(Path(entry).resolve() == Path(user_site).resolve()
       for entry in sys.path if entry for user_site in user_sites):
    raise RuntimeError(
        "This kernel includes ~/.local user-site packages. Register/select the "
        "qlora kernel with PYTHONNOUSERSITE=1, then restart it. See Training.md "
        "for kernel registration instructions."
    )
# Also isolate pip subprocesses when the kernel was launched with python -s.
os.environ["PYTHONNOUSERSITE"] = "1"
# Match the cluster's CUDA 12.6 toolkit. Override both settings for another toolkit.
torch_package = os.environ.get("QLORA_TORCH_PACKAGE", "torch==2.10.0+cu126")
torch_index = os.environ.get("QLORA_TORCH_INDEX_URL", "https://download.pytorch.org/whl/cu126")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-user", torch_package,
     "--index-url", torch_index], check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-user", "-U", "ipywidgets", torch_package,
     "transformers>=5.16.1", "torchvision", "kernels", "accelerate", "peft", "datasets", "bitsandbytes", "trl",
     "sentencepiece", "numpy>=2,<3", "pandas", "pyarrow", "scipy", "scikit-learn",
     "numexpr>=2.10.2", "bottleneck>=1.4.2", "ninja", "packaging",
     "huggingface_hub", "python-dotenv", "setuptools", "wheel"],
    check=True,
)

# Prefer the cluster/Conda toolkit; pip metadata is optional.
nvcc_candidates = []
for variable in ("CUDA_HOME", "CUDA_PATH"):
    if os.environ.get(variable):
        nvcc_candidates.append(Path(os.environ[variable]) / "bin" / "nvcc")
if path_nvcc := shutil.which("nvcc"):
    nvcc_candidates.append(Path(path_nvcc))
for package in ("nvidia-cuda-nvcc", "nvidia-cuda-nvcc-cu12", "nvidia-cuda-nvcc-cu11"):
    try:
        nvcc_dist = distribution(package)
    except PackageNotFoundError:
        continue
    nvcc_candidates.extend(
        Path(nvcc_dist.locate_file(file)) for file in nvcc_dist.files or []
        if str(file).endswith("/bin/nvcc")
    )
nvcc = next((path.resolve() for path in nvcc_candidates
             if path.is_file() and os.access(path, os.X_OK)), None)
if nvcc is not None:
    cuda_home = nvcc.parent.parent
    os.environ["CUDA_HOME"] = str(cuda_home)
    os.environ["PATH"] = str(nvcc.parent) + os.pathsep + os.environ.get("PATH", "")
    os.environ["CPATH"] = str(cuda_home / "include") + os.pathsep + os.environ.get("CPATH", "")
    print("CUDA compiler:", nvcc)
else:
    print("nvcc not found in this kernel. "
          "Training can use SDPA. To build optional extensions, "
          "load a matching cluster CUDA module before starting the notebook job.")

os.environ.setdefault("MAX_JOBS", "1")
# Let PyTorch infer GPU architecture unless TORCH_CUDA_ARCH_LIST was supplied.
system_cc = os.environ.get("CC") or shutil.which("gcc")
system_cxx = os.environ.get("CXX") or shutil.which("g++")
if system_cc:
    os.environ["CC"] = system_cc
if system_cxx:
    os.environ["CXX"] = system_cxx
if not system_cc or not system_cxx:
    print("Host compiler missing; load the cluster's GCC/G++ module before training.")

# A live kernel cannot safely switch to a different PyTorch binary.
if "torch" in sys.modules and sys.modules["torch"].__version__ != version("torch"):
    raise RuntimeError("PyTorch changed on disk. Restart the kernel and rerun setup.")

import torch

print("PyTorch:", torch.__version__, "| Built for CUDA:", torch.version.cuda,
      "| GPU available:", torch.cuda.is_available())
nvcc_version = None
if nvcc is not None:
    probe = subprocess.run([str(nvcc), "--version"], capture_output=True, text=True, check=False)
    match = re.search(r"\brelease\s+(\d+\.\d+)", probe.stdout)
    if probe.returncode == 0 and match:
        nvcc_version = match.group(1)
    print("Compiler CUDA:", nvcc_version or "unknown")
    if nvcc_version is None or nvcc_version != torch.version.cuda:
        print(f"Compiler diagnostic: nvcc CUDA {nvcc_version or 'unknown'} "
              f"does not match PyTorch CUDA {torch.version.cuda}. "
              "Use matching toolkit/PyTorch versions to compile optional kernels.")

print("Qwen3.5-27B uses SDPA for full attention and the PyTorch fallback for DeltaNet linear attention.")


Python: /common/home/projectgrps/CS480/CS480G3/.conda/envs/qlora/bin/python
Environment: /common/home/projectgrps/CS480/CS480G3/.conda/envs/qlora


Looking in indexes: https://download.pytorch.org/whl/cu126


  Using cached torchvision-0.29.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.7 kB)


INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached torchvision-0.29.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
  Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached torchvision-0.27.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchvision-0.26.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.0 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 39.6 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.6


    Uninstalling pandas-3.0.6:
      Successfully uninstalled pandas-3.0.6


CUDA compiler: /opt/apps/software/CUDA/12.4.0/bin/nvcc


PyTorch: 2.10.0+cu126 | Built for CUDA: 12.6 | GPU available: True
Compiler CUDA: 12.4
Compiler diagnostic: nvcc CUDA 12.4 does not match PyTorch CUDA 12.6. Use matching toolkit/PyTorch versions to compile optional kernels.
Qwen3.5-27B uses SDPA for full attention and the PyTorch fallback for DeltaNet linear attention.


## Imports and hardware

- Check the active Python environment, CUDA availability, and compiler.
- Model loading and training require one CUDA GPU. Set `RUN_TRAINING=False` to stop after data/tokenizer checks; these checks can run on CPU with the required packages installed.
- Load optional `.env` settings; Hugging Face libraries use `HF_TOKEN` from the environment when supplied.


In [2]:
import ast
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

# Set the compiler before importing PyTorch/Triton.
cc = os.environ.get("CC") or next(
    (path for name in ("gcc", "clang") if (path := shutil.which(name))),
    None,
)
if cc:
    os.environ["CC"] = cc

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from huggingface_hub import login, upload_folder
from IPython.display import display
from peft import LoraConfig, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

load_dotenv()

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Compiler:", cc)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB")


PyTorch: 2.10.0+cu126 | CUDA: True | Compiler: /opt/apps/software/GCCcore/13.3.0/bin/gcc
GPU: NVIDIA L40S | VRAM: 44.4 GiB


## Configuration

- Use `Qwen/Qwen3.5-27B`, the repository rubric dataset, and seeds `(42, 123, 2026)` for both splitting and training.
- Keep `r=48`, `alpha=96`, dropout `0.05`, and all training/generation settings from the original experiment. Only the LoRA target set expands.
- Batch size 1 × accumulation 8 gives effective batch size 8. `MAX_SEQ_LENGTH=None` sizes each seed from training/validation data only.
- `MODEL_REVISION=None` resolves the current checkpoint; use the original run's revision and recorded package versions for a controlled comparison.
- `RUN_TRAINING=True` and `RUN_TEST_EVALUATION=True` run three adapters and six evaluations. Greedy decoding uses `max(128, 8 × criterion count)` tokens with thinking disabled.
- Save to a dedicated output directory. Hugging Face upload remains opt-in.


In [3]:
MODEL_ID = "Qwen/Qwen3.5-27B"
MODEL_REVISION = None  # Set to a recorded commit hash to reproduce an earlier run.
DATASET_PATH = next(
    (str(parent / "Datasets/training_data_moh_eng_rubrics.csv")
     for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "Datasets/training_data_moh_eng_rubrics.csv").is_file()),
    "./Datasets/training_data_moh_eng_rubrics.csv",
)
OUTPUT_DIR = "./qwen35-27b-grading-qlora-inprojqkv-outproj-multiseed"
RUN_TRAINING = True
RUN_TEST_EVALUATION = True
RUN_HF_UPLOAD = False  # Opt in only when ready to publish one evaluated adapter.
UPLOAD_SEED = 42
UPLOAD_ALPHA = 96
HF_REPO_ID = "SmuFypTeam5/GradingQlora"
evaluation_completed = False
TRAINING_SEEDS = (42, 123, 2026)
MAX_SEQ_LENGTH = None  # Auto-size from formatted training/validation data.

LORA_R = 48
LORA_ALPHA = 96
LORA_ALPHAS = (LORA_ALPHA,)  # One adapter per seed.
if LORA_R != 48 or LORA_ALPHA != 96 or LORA_ALPHAS != (96,):
    raise ValueError("This experiment requires r=48 and alpha=96.")
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError("TRAINING_SEEDS must be nonempty and contain unique seeds.")
# Retain the original seven targets and add only the two DeltaNet projections.
LORA_TARGET_SUFFIXES = (
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
    "in_proj_qkv", "out_proj",
)
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size 8 on one GPU.
VALIDATIONS_PER_EPOCH = 4
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 1
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.0
GENERATION_MIN_TOKEN_BUDGET = 128
GENERATION_TOKENS_PER_CRITERION = 8

set_seed(TRAINING_SEEDS[0])


## Load the dataset

- Require prompt fields, binary labels, criterion maxima, and final-score metadata.
- Keep blank student answers empty and preserve literal `NA` text.


In [4]:
required_columns = [
    "question", "desired_answer", "student_answer", "criteria[]",
    "criteria_score[]", "criteria_max_score[]", "student_score",
]
if not Path(DATASET_PATH).is_file():
    raise FileNotFoundError(f"Rubric dataset not found: {DATASET_PATH}. Set DATASET_PATH in Configuration.")
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df.empty:
    raise ValueError("The dataset is empty.")
print("Rows:", len(df))
display(df.head())


Rows: 3778


,id,question,desired_answer,student_answer,criteria[],criteria_score[],criteria_max_score[],student_score
0,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,High risk problems are address in the prototyp...,"[""Answer mentions simulating behaviour of port...",[0],[1],0
1,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,To simulate portions of the desired final prod...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
2,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,A prototype program simulates the behaviors of...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
3,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,Defined in the Specification phase a prototype...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
4,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,It is used to let the users have a first idea ...,"[""Answer mentions simulating behaviour of port...",[0],[1],0


## Parse and validate rows

- Accept JSON/Python arrays and legacy quoted bracket arrays.
- Use label counts and `Answer` boundaries to preserve commas inside legacy criteria.
- Require nonempty criteria, aligned arrays, binary labels, and finite nonnegative score metadata.
- Keep recorded final scores even if they differ from the weighted criterion totals.


In [5]:
def parse_array(value, expected_count=None):
    """Parse structured arrays or legacy bracket text without losing criterion commas."""
    items = value if isinstance(value, list) else None
    if items is None:
        if not isinstance(value, str):
            raise TypeError(f"Expected an array or string, got {type(value).__name__}.")
        text = value.strip()
        # Unwrap quoted JSON/Python strings before handling legacy unquoted entries.
        for _ in range(3):
            try:
                parsed = json.loads(text)
            except json.JSONDecodeError:
                try:
                    parsed = ast.literal_eval(text)
                except (ValueError, SyntaxError):
                    break
            if isinstance(parsed, list):
                items = parsed
                break
            if not isinstance(parsed, str):
                raise ValueError(f"Expected a list, got {value!r}.")
            text = parsed.strip()
        if items is None:
            if not (text.startswith("[") and text.endswith("]")):
                raise ValueError(f"Expected a bracketed list, got {value!r}.")
            inner = text[1:-1].strip()
            if not inner:
                items = []
            elif expected_count == 1:
                items = [inner]
            elif re.search(r",\s*(?=Answer\s)", inner):
                items = re.split(r",\s*(?=Answer\s)", inner)
            else:
                items = next(csv.reader([inner], skipinitialspace=True))
            items = [item.strip() for item in items]
    if expected_count is not None and len(items) != expected_count:
        raise ValueError(f"Expected {expected_count} entries, parsed {len(items)}: {value!r}")
    return items


def parse_binary_scores(value):
    values = parse_array(value)
    if any(type(x) not in (int, str) or str(x) not in ("0", "1") for x in values):
        raise ValueError(f"Expected binary integers, got {value!r}")
    return [int(x) for x in values]


def parse_score_metadata(value):
    values = parse_array(value)
    if any(isinstance(x, bool) for x in values):
        raise ValueError("Criterion maxima cannot be booleans.")
    return [float(x) for x in values]


records = []
errors = []
for index, row in enumerate(df.to_dict("records")):
    try:
        scores = parse_binary_scores(row["criteria_score[]"])
        criteria = parse_array(row["criteria[]"], expected_count=len(scores))
        maxima = parse_score_metadata(row["criteria_max_score[]"])
        final_score = float(row["student_score"])
        if not criteria or any(not isinstance(c, str) or not c.strip() for c in criteria):
            raise ValueError("Criteria must be nonempty strings.")
        if len(criteria) != len(maxima):
            raise ValueError("Criterion, label, and maximum-score counts must match.")
        if any(not math.isfinite(x) or x < 0 for x in [*maxima, final_score]):
            raise ValueError("Score metadata must be finite and nonnegative.")
        if not str(row["question"]).strip() or not str(row["desired_answer"]).strip():
            raise ValueError("Question and desired answer must be nonempty.")
        records.append({
            "row_index": index,
            **{key: str(row[key]) for key in ("question", "desired_answer", "student_answer")},
            "criteria": criteria, "criteria_score": scores,
            "criteria_max_score": maxima, "student_score": final_score,
        })
    except (ValueError, TypeError, SyntaxError) as error:
        errors.append(f"Row {index}: {error}")
if errors:
    raise ValueError(f"{len(errors)} invalid rows:\n" + "\n".join(errors[:20]))
print(f"All {len(records)} rows passed validation.")
score_disagreements = sum(
    not math.isclose(
        sum(score * maximum for score, maximum in zip(row["criteria_score"], row["criteria_max_score"])),
        row["student_score"],
    )
    for row in records
)
print("Recorded final scores differing from criterion totals:", score_disagreements)


All 3778 rows passed validation.
Recorded final scores differing from criterion totals: 0


## Grading prompt

- Return only the ordered JSON array of binary decisions.
- Share one prompt builder between training and inference.


In [6]:
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Your task is to evaluate whether a student's answer fulfills each grading criterion.\n\n"
    "For each criterion:\n"
    "- Output 1 if the criterion is fulfilled.\n"
    "- Output 0 if the criterion is not fulfilled.\n\n"
    "Evaluate each criterion independently.\n\n"
    "Return ONLY a valid JSON array containing 0 or 1.\n"
    "The output array must contain exactly one value for each criterion, "
    "in the same order as the criteria provided.\n\n"
    "Do not include explanations.\n"
    "Do not include markdown.\n"
    "Do not include any text before or after the JSON array."
)

print(SYSTEM_PROMPT)


You are a strict grading assistant.

Your task is to evaluate whether a student's answer fulfills each grading criterion.

For each criterion:
- Output 1 if the criterion is fulfilled.
- Output 0 if the criterion is not fulfilled.

Evaluate each criterion independently.

Return ONLY a valid JSON array containing 0 or 1.
The output array must contain exactly one value for each criterion, in the same order as the criteria provided.

Do not include explanations.
Do not include markdown.
Do not include any text before or after the JSON array.


## Build examples

- Retain structured fields for evaluation; do not parse fields back out of prompt text.
- Keep the assistant completion separate from the prompt.


In [7]:
def make_prompt(question, desired_answer, student_answer, criteria):
    """Build the shared grading prompt without score metadata."""
    criteria_text = "\n".join(f"{i}. {text}" for i, text in enumerate(criteria, 1))
    user_text = (
        f"Question:\n{question}\n\n"
        f"Desired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]


dataset = Dataset.from_list([
    {
        **row,
        "prompt": make_prompt(row["question"], row["desired_answer"], row["student_answer"], row["criteria"]),
        "completion": [{"role": "assistant", "content": json.dumps(row["criteria_score"])}],
    }
    for row in records
])
print(dataset[0]["prompt"])


[{'role': 'system', 'content': "You are a strict grading assistant.\n\nYour task is to evaluate whether a student's answer fulfills each grading criterion.\n\nFor each criterion:\n- Output 1 if the criterion is fulfilled.\n- Output 0 if the criterion is not fulfilled.\n\nEvaluate each criterion independently.\n\nReturn ONLY a valid JSON array containing 0 or 1.\nThe output array must contain exactly one value for each criterion, in the same order as the criteria provided.\n\nDo not include explanations.\nDo not include markdown.\nDo not include any text before or after the JSON array."}, {'role': 'user', 'content': 'Question:\nWhat is the role of a prototype program in problem solving?\n\nDesired answer:\nTo simulate the behaviour of portions of the desired software product.\n\nStudent answer:\nHigh risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used to show a company that the software can be possibly programmed.\n

## Split by question for each seed

- Reuse `make_question_splits` from the multi-seed notebook, using the run seed to reshuffle all three question sets.
- Allocate approximately 80/10/10% of unique stripped questions; row proportions can differ. Require nonempty, disjoint splits covering every row.
- Preserve seeded question order. Base and QLoRA reuse the same split within a seed; held-out questions change across seeds.


In [8]:
question_keys = [row["question"].strip() for row in records]
unique_questions = list(dict.fromkeys(question_keys))


def make_question_splits(split_seed):
    shuffled_questions = unique_questions.copy()
    random.Random(split_seed).shuffle(shuffled_questions)
    n_questions = len(shuffled_questions)
    train_end = int(n_questions * 0.80)
    validation_end = train_end + int(n_questions * 0.10)
    question_splits = {
        "train": shuffled_questions[:train_end],
        "validation": shuffled_questions[train_end:validation_end],
        "test": shuffled_questions[validation_end:],
    }
    if any(not questions for questions in question_splits.values()):
        raise ValueError("Too few unique questions for nonempty 80/10/10 splits.")
    assert sum(map(len, question_splits.values())) == len(set.union(
        *(set(questions) for questions in question_splits.values())
    )) == n_questions
    indices_by_question = defaultdict(list)
    for index, question in enumerate(question_keys):
        indices_by_question[question].append(index)
    # Preserve the seeded question order in training, validation, and test.
    splits = {
        name: dataset.select([
            index for question in questions for index in indices_by_question[question]
        ])
        for name, questions in question_splits.items()
    }
    assert sum(map(len, splits.values())) == len(dataset)
    return {"split_seed": split_seed, "question_splits": question_splits, "splits": splits}


seed_data = {seed: make_question_splits(seed) for seed in TRAINING_SEEDS}
for seed, data in seed_data.items():
    print(f"Seed {seed} question split:")
    for name, split in data["splits"].items():
        print(f"  {name}: {len(split)} rows / {len(data['question_splits'][name])} questions")


Seed 42 question split:
  train: 2980 rows / 76 questions
  validation: 409 rows / 9 questions
  test: 389 rows / 10 questions
Seed 123 question split:
  train: 2889 rows / 76 questions
  validation: 611 rows / 9 questions
  test: 278 rows / 10 questions
Seed 2026 question split:
  train: 3113 rows / 76 questions
  validation: 269 rows / 9 questions
  test: 396 rows / 10 questions


## Tokenizer

- Use the base-model notebook's `AutoProcessor` loader and its text tokenizer. Copy its chat template to the tokenizer so pre-tokenization and inference render the same text as the processor.
- Share this tokenizer/template across training, validation, base evaluation, and QLoRA evaluation, with thinking disabled.
- Use the tokenizer's resolved commit hash for every model load when available; record the revision in `experiment.json`.


In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer = getattr(processor, "tokenizer", processor)
if getattr(processor, "chat_template", None):
    tokenizer.chat_template = processor.chat_template
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
# Prefer the resolved commit hash when the tokenizer provides it.
MODEL_REVISION = tokenizer.init_kwargs.get("_commit_hash") or MODEL_REVISION
print("Tokenizer:", tokenizer.__class__.__name__)


Tokenizer: Qwen3_5Tokenizer


## Tokenize and verify completion masks for every seed

- Reuse the multi-seed notebook's tokenization loop; prepare each seed's training/validation examples once.
- Qwen's native template includes the empty `<think>...</think>` block in both the non-thinking generation prompt and full assistant conversation.
- Verify the generation prompt is an exact prefix of the full conversation. Mask prompt labels with `-100` and supervise completion tokens only.
- Auto-size each seed's sequence limit from training/validation only, or reject examples above an explicit cap. Test examples do not set training limits.


In [10]:
def tokenize_example(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"], tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    input_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=True, return_dict=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    boundary = len(prompt_ids)
    if input_ids[:boundary] != prompt_ids or boundary >= len(input_ids):
        raise ValueError(f"Invalid prompt/completion boundary at row {example['row_index']}.")
    return {"input_ids": input_ids, "labels": [-100] * boundary + input_ids[boundary:]}


for seed, data in seed_data.items():
    formatted_train, formatted_validation = (
        data["splits"][name].map(
            tokenize_example, remove_columns=data["splits"][name].column_names,
            desc=f"Seed {seed}: tokenize {name}",
        )
        for name in ("train", "validation")
    )
    lengths = [len(ids) for split in (formatted_train, formatted_validation) for ids in split["input_ids"]]
    measured_max_length = max(lengths)
    sequence_limit = MAX_SEQ_LENGTH if MAX_SEQ_LENGTH is not None else math.ceil(measured_max_length / 32) * 32
    if sequence_limit < measured_max_length:
        raise ValueError(
            f"Seed {seed}: {sum(length > sequence_limit for length in lengths)} examples exceed "
            f"{sequence_limit} tokens; longest is {measured_max_length}. Increase MAX_SEQ_LENGTH or use None."
        )
    data.update({
        "formatted_train": formatted_train, "formatted_validation": formatted_validation,
        "sequence_limit": sequence_limit,
    })
    print(f"Seed {seed} training/validation token lengths:", pd.Series(lengths).describe().round(1).to_dict())
    print("Sequence limit:", sequence_limit)
    print("Example supervised completion:", tokenizer.decode(
        [label for label in formatted_train[0]["labels"] if label != -100], skip_special_tokens=False,
    ))


Seed 42: tokenize train:   0%|          | 0/2980 [00:00<?, ? examples/s]

Seed 42: tokenize validation:   0%|          | 0/409 [00:00<?, ? examples/s]

Seed 42 training/validation token lengths: {'count': 3389.0, 'mean': 296.1, 'std': 108.5, 'min': 178.0, '25%': 221.0, '50%': 258.0, '75%': 331.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 1, 1]<|im_end|>



Seed 123: tokenize train:   0%|          | 0/2889 [00:00<?, ? examples/s]

Seed 123: tokenize validation:   0%|          | 0/611 [00:00<?, ? examples/s]

Seed 123 training/validation token lengths: {'count': 3500.0, 'mean': 295.5, 'std': 105.2, 'min': 178.0, '25%': 222.0, '50%': 266.0, '75%': 327.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 1, 1]<|im_end|>



Seed 2026: tokenize train:   0%|          | 0/3113 [00:00<?, ? examples/s]

Seed 2026: tokenize validation:   0%|          | 0/269 [00:00<?, ? examples/s]

Seed 2026 training/validation token lengths: {'count': 3382.0, 'mean': 296.6, 'std': 108.3, 'min': 179.0, '25%': 221.0, '50%': 259.0, '75%': 331.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 0]<|im_end|>



## Quantisation and model loading

- Load a fresh Qwen3.5-27B model per seed with `AutoModelForMultimodalLM`, the same loader used in the base-model notebook. All inputs in this experiment are text.
- Keep Scaling's NF4, double quantisation, BF16/FP16 compute, SDPA, and single-GPU placement. Disable caching in both the wrapper and text configuration during training.
- Qwen's linear-attention layers use the PyTorch DeltaNet fallback with the source notebook's dependency setup.
- Release each seed's model before loading the next seed.


In [11]:
def load_quantized_base_model():
    """Load the base benchmark's Qwen3.5-27B checkpoint in NF4 on one GPU."""
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA training requires a CUDA GPU.")
    if not cc:
        raise RuntimeError("Install a C compiler with the dependency cell, then restart the kernel.")
    if int(os.environ.get("WORLD_SIZE", "1")) != 1:
        raise RuntimeError("This notebook's batch and validation schedule assume one training GPU.")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, quantization_config=bnb_config, device_map={"": 0},
        dtype=compute_dtype, attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model.config.get_text_config().use_cache = False
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print("Loaded in 4-bit:", model.is_loaded_in_4bit, "| Compute dtype:", compute_dtype)
    return model, compute_dtype


## Attach LoRA adapters

- Retain `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, and `down_proj`; add `linear_attn.in_proj_qkv` and `linear_attn.out_proj`.
- Restrict targets to linear modules under `model.language_model.layers`. Require both new projections in every Gated DeltaNet layer and print target counts before training.
- Keep rank 48, alpha 96, dropout 0.05, `bias="none"`, and `use_rslora=False`. Base weights and all untargeted modules remain frozen.
- Projection names follow the [Transformers Qwen3.5 implementation](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_5/modeling_qwen3_5.py).


In [12]:
def attach_fresh_lora(model, alpha):
    """Attach the original targets plus DeltaNet QKV/output projections."""
    for param in model.parameters():
        param.requires_grad_(False)
    candidate_suffixes = set(LORA_TARGET_SUFFIXES)
    target_modules = [
        name for name, module in model.named_modules()
        if name.startswith("model.language_model.layers.") and isinstance(module, torch.nn.Linear)
        and name.rsplit(".", 1)[-1] in candidate_suffixes
    ]
    if not target_modules:
        raise RuntimeError("No Qwen3.5-27B language projection targets found under model.language_model.layers; inspect model.named_modules().")
    # Fail before training if either new projection is absent in a DeltaNet layer.
    deltanet_layers = [
        name for name, _ in model.named_modules()
        if name.startswith("model.language_model.layers.") and name.endswith(".linear_attn")
    ]
    required_targets = {
        f"{name}.{suffix}" for name in deltanet_layers
        for suffix in ("in_proj_qkv", "out_proj")
    }
    missing_targets = sorted(required_targets - set(target_modules))
    if not deltanet_layers or missing_targets:
        raise RuntimeError(f"Missing Gated DeltaNet projection targets: {missing_targets or 'no linear_attn layers found'}")
    print("LoRA projection layers:", len(target_modules))
    print("Targets by projection:", {
        suffix: sum(name.rsplit(".", 1)[-1] == suffix for name in target_modules)
        for suffix in LORA_TARGET_SUFFIXES
    })
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=alpha, lora_dropout=LORA_DROPOUT,
        use_rslora=False, bias="none", task_type="CAUSAL_LM", target_modules=target_modules,
    ))
    model.print_trainable_parameters()
    return model


## Training configuration

- Reuse the scaling notebook's loss-based training controls, with the multi-seed notebook's per-seed schedule calculation.
- Each seed gets approximately four validation/save steps per epoch, based on its own training row count. Use that seed's schedule and sequence limit.
- Use paged 8-bit AdamW, mixed precision, non-reentrant gradient checkpointing, and completion-only NLL without packing or activation offloading.
- Plateau LR reduction, early stopping, and checkpoint restoration minimize `eval_loss`. Test metrics never select checkpoints.
- Pass the run seed explicitly to both Trainer's `seed` and `data_seed`. Save checkpoints in that seed's alpha_96 directory.


In [13]:
for seed, data in seed_data.items():
    microbatches_per_epoch = math.ceil(len(data["formatted_train"]) / TRAIN_BATCH_SIZE)
    optimizer_steps_per_epoch = math.ceil(microbatches_per_epoch / GRADIENT_ACCUMULATION_STEPS)
    data["optimizer_steps_per_epoch"] = optimizer_steps_per_epoch
    data["quarter_epoch_steps"] = max(1, round(optimizer_steps_per_epoch / VALIDATIONS_PER_EPOCH))
    print(f"Seed {seed}: {optimizer_steps_per_epoch} optimizer steps per epoch; "
          f"evaluate/save every {data['quarter_epoch_steps']} optimizer steps")


def make_training_args(run_dir, compute_dtype, training_seed, data):
    """Build shared controls with this seed's data and independent run state."""
    quarter_epoch_steps = data["quarter_epoch_steps"]
    return SFTConfig(
        output_dir=str(run_dir),
        seed=training_seed,
        data_seed=training_seed,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optim="paged_adamw_8bit",
        lr_scheduler_type="reduce_lr_on_plateau",
        lr_scheduler_kwargs={
            "mode": "min", "factor": LR_REDUCTION_FACTOR,
            "patience": LR_PATIENCE, "min_lr": MIN_LEARNING_RATE,
        },
        bf16=(compute_dtype == torch.bfloat16),
        fp16=(compute_dtype == torch.float16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        activation_offloading=False,
        loss_type="nll",
        completion_only_loss=True,
        max_length=data["sequence_limit"],
        packing=False,
        padding_free=False,
        logging_steps=max(1, quarter_epoch_steps // 2),
        save_strategy="steps",
        save_steps=quarter_epoch_steps,
        eval_strategy="steps",
        eval_steps=quarter_epoch_steps,
        prediction_loss_only=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        report_to="none",
    )


Seed 42: 373 optimizer steps per epoch; evaluate/save every 93 optimizer steps
Seed 123: 362 optimizer steps per epoch; evaluate/save every 90 optimizer steps
Seed 2026: 390 optimizer steps per epoch; evaluate/save every 98 optimizer steps


## Create the trainer

- Create independent trainer, optimizer, scheduler, and early-stopping state for each seed.
- Reuse that seed's prepared examples and tokenizer. Validation computes loss only, as in the scaling notebook.


In [14]:
def create_trainer(model, run_dir, compute_dtype, training_seed, data):
    """Create independent training state for the current adapter."""
    return SFTTrainer(
        model=model, args=make_training_args(run_dir, compute_dtype, training_seed, data),
        train_dataset=data["formatted_train"], eval_dataset=data["formatted_validation"],
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )],
    )


## Check the collated loss mask

- Before training each configuration, inspect up to two examples using its trainer's data collator.
- Prompt and padding labels must be `-100`; completion tokens must remain supervised.


In [15]:
def verify_completion_masks(trainer):
    """Verify that the collator supervises completion tokens only."""
    samples = [trainer.train_dataset[i] for i in range(min(2, len(trainer.train_dataset)))]
    batch = trainer.data_collator(samples)
    for index, sample in enumerate(samples):
        labels = batch["labels"][index]
        length = len(sample["labels"])
        assert labels[:length].tolist() == sample["labels"], "Collator changed the completion mask."
        assert (labels[length:] == -100).all(), "Padding contributes to loss."
        assert (labels[:length] == -100).any() and (labels[:length] != -100).any()
    print("Prompt/completion/padding masks verified.")


## Train and save

- Train the current adapter, restore its lowest-validation-loss checkpoint, and save the adapter and tokenizer in `seed_<seed>/alpha_<value>/`.
- Save training metrics, loss history, and best-checkpoint metadata alongside the adapter.
- Release optimizer, scheduler, and gradient storage before inference, including cleanup after a failed training run.


In [16]:
def train_and_save(model, run_dir, compute_dtype, training_seed, data):
    """Save the validation-selected adapter and release training storage."""
    trainer = None
    try:
        trainer = create_trainer(model, run_dir, compute_dtype, training_seed, data)
        verify_completion_masks(trainer)
        train_result = trainer.train()
        trainer.save_model(str(run_dir))
        tokenizer.save_pretrained(str(run_dir))
        (run_dir / "training_metrics.json").write_text(json.dumps(train_result.metrics, indent=2) + "\n")
        pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "training_history.csv", index=False)
        (run_dir / "best_checkpoint.json").write_text(json.dumps({
            "checkpoint": trainer.state.best_model_checkpoint,
            "validation_loss": trainer.state.best_metric,
        }, indent=2) + "\n")
        print("Saved best adapter and tokenizer to:", run_dir)
        display(train_result.metrics)
        return trainer.model
    finally:
        if trainer is not None:
            trainer.model.zero_grad(set_to_none=True)
            trainer.optimizer = None
            trainer.lr_scheduler = None
            trainer.accelerator.free_memory()
        trainer = None
        gc.collect()
        torch.cuda.empty_cache()


## Prepare inference

- Reuse the saved adapter's model after training storage has been released.
- Disable gradient checkpointing, enable the KV cache, and switch to evaluation mode.


In [17]:
def prepare_inference(model):
    """Switch the trained model to cached inference."""
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.config.get_text_config().use_cache = True
    model.eval()


## Prediction helper

- Use the shared prompt, greedy decoding, and no thinking.
- Accept only integer `0`/`1` arrays of the correct length; reject floats and booleans.
- If JSON decoding fails, recover the last binary array of the expected length from surrounding text and report it as `recovered`.


In [18]:
def parse_prediction(raw_output, expected_count):
    """Validate binary decisions, recovering an embedded array when needed."""
    recovered = False
    try:
        scores = json.loads(raw_output)
    except json.JSONDecodeError:
        candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw_output)
        scores = next((
            parsed for candidate in reversed(candidates)
            if len(parsed := json.loads(candidate)) == expected_count
        ), None)
        recovered = scores is not None
    error = None
    if not isinstance(scores, list):
        error = "Output was not a JSON array."
    elif len(scores) != expected_count:
        error = f"Expected {expected_count} scores, received {len(scores)}."
    elif any(type(score) is not int or score not in (0, 1) for score in scores):
        error = "Scores must be integer 0 or 1."
    return {
        "valid": error is None, "raw_output": raw_output,
        "scores": scores if error is None else None,
        "error": error, "recovered": recovered,
    }


@torch.inference_mode()
def predict_criteria_scores(question, desired_answer, student_answer, criteria, prediction_model):
    """Generate one greedy rubric prediction with the shared parser."""
    inputs = tokenizer.apply_chat_template(
        make_prompt(question, desired_answer, student_answer, criteria),
        tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
    ).to(prediction_model.get_input_embeddings().weight.device)
    generated = prediction_model.generate(
        **inputs, do_sample=False,
        max_new_tokens=max(GENERATION_MIN_TOKEN_BUDGET, len(criteria) * GENERATION_TOKENS_PER_CRITERION),
    )
    answer_ids = generated[0, inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return parse_prediction(raw_output, len(criteria))


## Manual prediction

- Run the same illustrative example after each adapter finishes training, including when test evaluation is disabled.
- The expected rubric array is `[1, 0, 0]`. This example is separate from the held-out metrics.


In [19]:
def show_manual_prediction(model):
    """Print an illustrative prediction outside the test-set metrics."""
    result = predict_criteria_scores(
        question="Explain why HTTPS is more secure than HTTP.",
        desired_answer="HTTPS encrypts network traffic using TLS.",
        student_answer="HTTPS protects transmitted data using encryption.",
        criteria=["Mentions encryption", "Mentions TLS", "Claims HTTPS sends plaintext"],
        prediction_model=model,
    )
    print(result)


## Evaluate the held-out test set

- Use the scaling notebook's evaluator, prompt, parser, and decoding procedure for every run; within each seed, the Qwen3.5-27B base and QLoRA adapter share its test split.
- Compute the final score as the sum of each predicted binary decision × its criterion maximum, compared with the recorded `student_score`.
- Invalid outputs count as failures for exact-array and final-score accuracy; criterion metrics, MAE, RMSE, Pearson, and QWK use valid outputs only.
- Pearson is undefined for fewer than two valid predictions or constant true/predicted scores. Other undefined metrics remain `NaN`.
- Save every metric, including invalid/recovered-output rates, plus raw predictions and per-example errors.


In [20]:
def evaluate_current_model(dataset, model_name, prediction_model):
    """Score one model state with the shared invalid-output and final-score rules."""
    criterion_true = []
    criterion_pred = []
    exact_correct = 0
    invalid_count = 0
    final_score_errors = []
    final_score_exact = 0
    final_score_within_one = 0
    valid_true_final_scores = []
    valid_predicted_final_scores = []
    rows = []

    for example in dataset:
        true_scores = example["criteria_score"]
        max_scores = example["criteria_max_score"]
        true_final_score = example["student_score"]
        prediction = predict_criteria_scores(
            example["question"], example["desired_answer"],
            example["student_answer"], example["criteria"], prediction_model=prediction_model,
        )

        # Invalid arrays are failures for response-level exact matching.
        is_valid = prediction["valid"]
        pred_scores = prediction["scores"] if is_valid else None
        is_exact = bool(is_valid and pred_scores == true_scores)
        exact_correct += int(is_exact)
        invalid_count += int(not is_valid)

        # Criterion metrics require aligned arrays, so invalid arrays are omitted.
        if is_valid:
            criterion_true.extend(true_scores)
            criterion_pred.extend(pred_scores)

            # Convert binary criterion decisions into the final numeric score.
            predicted_final_score = sum(
                score * maximum for score, maximum in zip(pred_scores, max_scores)
            )
            score_error = abs(predicted_final_score - true_final_score)
            final_score_errors.append(score_error)
            final_score_exact += int(score_error == 0)
            final_score_within_one += int(score_error <= 1)
            valid_true_final_scores.append(true_final_score)
            valid_predicted_final_scores.append(predicted_final_score)
        else:
            predicted_final_score = None
            score_error = None

        rows.append({
            "model": model_name,
            "row_index": example["row_index"],
            "raw_output": prediction["raw_output"],
            "recovered": prediction["recovered"],
            "error": prediction["error"],
            "ground_truth": true_scores,
            "prediction": pred_scores if is_valid else prediction["raw_output"],
            "valid": is_valid,
            "exact_match": is_exact,
            "true_final_score": true_final_score,
            "predicted_final_score": predicted_final_score,
            "final_score_absolute_error": score_error,
        })

    total = len(dataset)
    precision, recall, f1, _ = (
        precision_recall_fscore_support(criterion_true, criterion_pred, average="binary", zero_division=0)
        if criterion_true else (float("nan"),) * 4
    )
    if (
        len(valid_true_final_scores) > 1
        and len(set(valid_true_final_scores)) > 1
        and len(set(valid_predicted_final_scores)) > 1
    ):
        final_score_pearson = float(
            np.corrcoef(valid_true_final_scores, valid_predicted_final_scores)[0, 1]
        )
    else:
        final_score_pearson = float("nan")
    if valid_true_final_scores:
        final_score_qwk = float(cohen_kappa_score(
            valid_true_final_scores, valid_predicted_final_scores, weights="quadratic"
        ))
    else:
        final_score_qwk = float("nan")

    metrics = {
        "model": model_name,
        "examples": total,
        "valid_examples": total - invalid_count,
        "recovered_output_rate": sum(row["recovered"] for row in rows) / total if total else 0.0,
        "invalid_output_rate": invalid_count / total if total else 0.0,
        "exact_match_accuracy": exact_correct / total if total else 0.0,
        "per_criterion_accuracy": accuracy_score(criterion_true, criterion_pred) if criterion_true else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        # Invalid generations count as incorrect for the two score accuracies.
        "final_score_exact_accuracy": final_score_exact / total if total else 0.0,
        "final_score_within_one_accuracy": final_score_within_one / total if total else 0.0,
        # MAE, RMSE, Pearson, and QWK describe valid numeric predictions only.
        "final_score_mae": (
            sum(final_score_errors) / len(final_score_errors)
            if final_score_errors else float("nan")
        ),
        "final_score_rmse": (
            (sum(error ** 2 for error in final_score_errors) / len(final_score_errors)) ** 0.5
            if final_score_errors else float("nan")
        ),
        "final_score_pearson": final_score_pearson,
        "final_score_qwk": final_score_qwk,
    }
    return metrics, rows


## Run baseline and extended QLoRA across all seeds

- Train one adapter for each seed: 42, 123, and 2026, with rank 48 and alpha 96.
- Reset the seed before fresh model/adapter creation. Reuse the original split, tokenisation, sequence limit, validation schedule, and lowest-validation-loss selection.
- Evaluate **Baseline** with the adapter disabled and extended **QLoRA** with it enabled on the same test rows.
- Save the experiment manifest, adapter, metrics, and raw predictions; release GPU memory between seeds.
- Mark evaluation complete only after all six results pass validation. Each run uses a fresh directory.


In [21]:
def save_evaluation(run_dir, label, metrics, rows):
    """Persist complete metrics and raw predictions for one model state."""
    pd.DataFrame([metrics]).to_csv(run_dir / f"{label}_metrics.csv", index=False)
    pd.DataFrame(rows).to_json(run_dir / f"{label}_predictions.jsonl", orient="records", lines=True)


def run_configuration(alpha, training_seed, experiment_dir, include_baseline=False):
    """Train one seed/alpha pair on its seed's split and release GPU memory."""
    seed_dir = experiment_dir / f"seed_{training_seed}"
    seed_dir.mkdir(exist_ok=True)
    run_dir = seed_dir / f"alpha_{alpha}"
    data = seed_data[training_seed]
    test_dataset = data["splits"]["test"]
    run_dir.mkdir(exist_ok=False)
    model = None
    results, predictions = [], []
    try:
        set_seed(training_seed)
        model, compute_dtype = load_quantized_base_model()
        model = attach_fresh_lora(model, alpha)
        model = train_and_save(model, run_dir, compute_dtype, training_seed, data)
        prepare_inference(model)
        show_manual_prediction(model)
        if RUN_TEST_EVALUATION:
            if include_baseline:
                with model.disable_adapter():
                    metrics, rows = evaluate_current_model(test_dataset, "Baseline", prediction_model=model)
                metrics = {**metrics, "seed": training_seed, "alpha": None}
                rows = [{**row, "seed": training_seed, "alpha": None} for row in rows]
                save_evaluation(seed_dir, "baseline", metrics, rows)
                results.append(metrics)
                predictions.extend(rows)
            metrics, rows = evaluate_current_model(test_dataset, "QLoRA", prediction_model=model)
            metrics = {**metrics, "seed": training_seed, "alpha": alpha}
            rows = [{**row, "seed": training_seed, "alpha": alpha} for row in rows]
            save_evaluation(run_dir, "test", metrics, rows)
            results.append(metrics)
            predictions.extend(rows)
        return results, predictions
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()



def validate_seed_results(results_df, expected_seeds):
    """Require one Qwen3.5-27B baseline and one QLoRA adapter on every seed's complete test set."""
    expected_models = ["Baseline", "QLoRA"]
    expected = {(seed, model) for seed in expected_seeds for model in expected_models}
    if results_df.empty or not {"seed", "model", "examples"}.issubset(results_df.columns):
        raise ValueError("Seed evaluation results are empty or missing identifying columns.")
    if results_df.duplicated(["seed", "model"]).any():
        raise ValueError("Duplicate evaluation results for a seed/model pair.")
    actual = set(results_df[["seed", "model"]].itertuples(index=False, name=None))
    if actual != expected:
        raise ValueError(f"Incomplete results. Missing: {sorted(expected - actual)}; unexpected: {sorted(actual - expected)}")
    for seed in expected_seeds:
        subset = results_df.loc[results_df["seed"] == seed]
        if not subset["examples"].eq(len(seed_data[seed]["splits"]["test"])).all():
            raise ValueError(f"Seed {seed}: every model must evaluate the complete shared test set.")


all_metrics, all_prediction_rows = [], []
experiment_dir = None
evaluation_completed = False
if RUN_TRAINING:
    experiment_dir = Path(OUTPUT_DIR) / datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S-%fZ")
    experiment_dir.mkdir(parents=True, exist_ok=False)
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    experiment_manifest = {
        "model": MODEL_ID, "model_revision": MODEL_REVISION,
        "dataset_path": str(Path(DATASET_PATH).resolve()),
        "dataset_sha256": hashlib.sha256(Path(DATASET_PATH).read_bytes()).hexdigest(),
        "training_seeds": TRAINING_SEEDS, "split_seed_policy": "same as training seed",
        "question_split_fractions": {"train": 0.8, "validation": 0.1, "test": 0.1},
        "lora_rank": LORA_R,
        "lora_target_suffixes": LORA_TARGET_SUFFIXES,
        "configurations": [{"alpha": alpha, "scaling": alpha / LORA_R} for alpha in LORA_ALPHAS],
        "seed_splits": {
            str(seed): {
                "split_seed": data["split_seed"], "question_splits": data["question_splits"],
                "split_row_indices": {
                    name: [row["row_index"] for row in split] for name, split in data["splits"].items()
                },
                "row_counts": {name: len(split) for name, split in data["splits"].items()},
                "sequence_limit": data["sequence_limit"],
                "optimizer_steps_per_epoch": data["optimizer_steps_per_epoch"],
                "quarter_epoch_steps": data["quarter_epoch_steps"],
                "training_args_by_alpha": {
                    str(alpha): make_training_args(
                        experiment_dir / f"seed_{seed}" / f"alpha_{alpha}", compute_dtype, seed, data,
                    ).to_dict()
                    for alpha in LORA_ALPHAS
                },
            }
            for seed, data in seed_data.items()
        },
        "lora_dropout": LORA_DROPOUT, "use_rslora": False,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
        "generation": {
            "do_sample": False, "enable_thinking": False,
            "max_new_tokens_floor": GENERATION_MIN_TOKEN_BUDGET,
            "tokens_per_criterion": GENERATION_TOKENS_PER_CRITERION,
        },
        "package_versions": {name: version(name) for name in (
            "torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets", "numpy", "pandas", "scikit-learn",
        )},
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "cuda_version": torch.version.cuda,
    }
    (experiment_dir / "experiment.json").write_text(json.dumps(experiment_manifest, indent=2, default=str) + "\n")
    print("Experiment directory:", experiment_dir)
    for training_seed in TRAINING_SEEDS:
        for index, alpha in enumerate(LORA_ALPHAS):
            print(f"Starting seed {training_seed}, α={alpha}, r={LORA_R}, scaling={alpha / LORA_R:g}")
            metrics, rows = run_configuration(
                alpha, training_seed, experiment_dir, include_baseline=(index == 0),
            )
            all_metrics.extend(metrics)
            all_prediction_rows.extend(rows)
            if metrics:
                pd.DataFrame(all_metrics).to_csv(experiment_dir / "qwen35_27b_evaluation_comparison.csv", index=False)
    if RUN_TEST_EVALUATION:
        validate_seed_results(pd.DataFrame(all_metrics), TRAINING_SEEDS)
        evaluation_completed = True
else:
    print("Data checks complete. Set RUN_TRAINING=True to train Qwen3.5-27B QLoRA across all three seeds.")


Experiment directory: qwen35-27b-grading-qlora-inprojqkv-outproj-multiseed/run-20261023T050357-563976Z
Starting seed 42, α=96, r=48, scaling=2


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Loaded in 4-bit: True | Compute dtype: torch.bfloat16
LoRA projection layers: 352
Targets by projection: {'q_proj': 16, 'k_proj': 16, 'v_proj': 16, 'o_proj': 16, 'gate_proj': 64, 'up_proj': 64, 'down_proj': 64, 'in_proj_qkv': 48, 'out_proj': 48}


trainable params: 300,417,024 || all params: 27,657,145,584 || trainable%: 1.0862


Truncating train dataset:   0%|          | 0/2980 [00:00<?, ? examples/s]

...

Prompt/completion/padding masks verified.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
98,0.065109,0.089701,0.081499,0.961043,235047.000000
196,0.054115,0.077198,0.076856,0.964997,469970.000000
294,0.055374,0.090575,0.075506,0.955847,709036.000000
392,0.054516,0.077221,0.077468,0.964929,945328.000000
490,0.025183,0.088335,0.052796,0.965588,1185591.000000
588,0.022325,0.098523,0.044666,0.963113,1423005.000000


Saved best adapter and tokenizer to: qwen35-27b-grading-qlora-inprojqkv-outproj-multiseed/run-20261023T050357-563976Z/seed_2026/alpha_96


{'train_runtime': 3870.1719,
 'train_samples_per_second': 1.213,
 'train_steps_per_second': 0.152,
 'total_flos': 2.2525963932187728e+17,
 'train_loss': 0.0475921428122488,
 'epoch': 1.5088339222614842}

{'valid': True, 'raw_output': '[1, 0, 0]', 'scores': [1, 0, 0], 'error': None, 'recovered': False}


## Inspect predictions

- Preview the first 20 predictions, including seed, alpha, and model identifiers. Baseline alpha is empty.
- Retain raw output, parsed decisions, validity/recovery status, and final-score errors. Complete JSONL files are saved separately for each seed/model.


In [22]:
if evaluation_completed:
    final_score_predictions_df = pd.DataFrame(all_prediction_rows)
    display(final_score_predictions_df[[
        "seed", "alpha", "model", "row_index", "ground_truth", "prediction", "raw_output",
        "true_final_score", "predicted_final_score", "final_score_absolute_error", "valid", "recovered",
    ]].head(20))


,seed,alpha,model,row_index,ground_truth,prediction,raw_output,true_final_score,predicted_final_score,final_score_absolute_error,valid,recovered
0,42,NaN,Baseline,1834,[1],[0],[0],1.0,0.0,1.0,True,False
1,42,NaN,Baseline,1835,[0],[0],[0],0.0,0.0,0.0,True,False
2,42,NaN,Baseline,1836,[1],[0],[0],1.0,0.0,1.0,True,False
3,42,NaN,Baseline,1837,[0],[0],[0],0.0,0.0,0.0,True,False
4,42,NaN,Baseline,1838,[0],[0],[0],0.0,0.0,0.0,True,False
5,42,NaN,Baseline,1839,[0],[0],[0],0.0,0.0,0.0,True,False
6,42,NaN,Baseline,1840,[1],[0],[0],1.0,0.0,1.0,True,False
7,42,NaN,Baseline,1841,[1],[1],[1],1.0,1.0,0.0,True,False
8,42,NaN,Baseline,1842,[0],[0],[0],0.0,0.0,0.0,True,False
9,42,NaN,Baseline,1843,[0],[0],[0],0.0,0.0,0.0,True,False


## Baseline versus extended QLoRA across seeds

- **Table 1:** Baseline and extended QLoRA for seeds 42, 123, and 2026 (six rows).
- **Table 2:** equal-weight means across seeds. Also export sample standard deviations (`ddof=1`) and per-seed QLoRA-minus-baseline differences.
- The concise comparison reports QWK, Pearson, RMSE, MAE, and exact final-score accuracy. Accuracy counts invalid outputs as failures; the other four metrics use valid predictions only. Check validity rates in the full tables.
- Highlight tied winners using unrounded means; display four decimals and retain full precision in CSV exports. Undefined metrics remain `NaN` and are skipped when averaging.
- Preserve the original CSV names and schemas within this experiment's dedicated output directory. `experiment.json` and each adapter's configuration record the expanded targets.


In [23]:
METRIC_COLUMNS = {
    "final_score_qwk": "QWK",
    "final_score_pearson": "Pearson",
    "final_score_rmse": "RMSE",
    "final_score_mae": "MAE",
    "final_score_exact_accuracy": "Accuracy",
}
HIGHER_IS_BETTER = ["QWK", "Pearson", "Accuracy"]
LOWER_IS_BETTER = ["RMSE", "MAE"]


def build_evaluation_tables(all_metrics):
    """Keep the multi-seed table structure and average only evaluation metrics."""
    results = pd.DataFrame(all_metrics)
    validate_seed_results(results, TRAINING_SEEDS)
    metric_columns = [column for column in results if column not in {"seed", "alpha", "model"}]
    row_order = pd.MultiIndex.from_product(
        [TRAINING_SEEDS, ["Baseline", "QLoRA"]],
        names=["seed", "model"],
    )
    ordered = results.set_index(["seed", "model"]).reindex(row_order)
    per_seed = ordered[metric_columns]
    model_order = ["Baseline", "QLoRA"]
    grouped = per_seed.groupby(level="model", sort=False)
    averages = grouped.mean().reindex(model_order)
    standard_deviations = grouped.std(ddof=1).reindex(model_order)
    return per_seed, averages, standard_deviations


def display_metric_table(table, title):
    print(title)
    with pd.option_context(
        "display.max_rows", None, "display.max_columns", None,
        "display.float_format", "{:.4f}".format,
    ):
        display(table)


def metric_winners(table):
    """Return all tied winners per metric, omitting undefined values."""
    winners = {}
    for metric in table.columns:
        values = table[metric].dropna()
        best = values.max() if metric in HIGHER_IS_BETTER else values.min()
        winners[metric] = values.index[values.eq(best)].tolist()
    return winners


if evaluation_completed:
    results_df, average_results_df, standard_deviations_df = build_evaluation_tables(all_metrics)
    display_metric_table(results_df, "Table 1: Performance Across Testing Seeds")
    display_metric_table(average_results_df, "Table 2: Average Performance Across Testing Seeds")
    results_df.to_csv(experiment_dir / "per_seed_results.csv")
    average_results_df.to_csv(experiment_dir / "average_results.csv")
    standard_deviations_df.to_csv(experiment_dir / "across_seed_standard_deviations.csv")
    for seed in TRAINING_SEEDS:
        seed_results = results_df.xs(seed, level="seed")
        differences = seed_results.drop(index="Baseline").subtract(seed_results.loc["Baseline"], axis="columns")
        differences.to_csv(experiment_dir / f"seed_{seed}_comparison_to_baseline.csv")
    comparison_df = average_results_df[list(METRIC_COLUMNS)].rename(columns=METRIC_COLUMNS)
    comparison_std_df = standard_deviations_df[list(METRIC_COLUMNS)].rename(columns=METRIC_COLUMNS)
    print("Mean Qwen3.5-27B baseline versus extended QLoRA (r=48, alpha=96; +in_proj_qkv, +out_proj); highlighted cells are metric winners.")
    display(
        comparison_df.style.format("{:.4f}", na_rep="NaN")
        .highlight_max(subset=HIGHER_IS_BETTER, color="#d9ead3")
        .highlight_min(subset=LOWER_IS_BETTER, color="#d9ead3")
    )
    comparison_df.to_csv(experiment_dir / "qwen35_27b_comparison.csv")
    comparison_std_df.to_csv(experiment_dir / "qwen35_27b_comparison_std.csv")
    display_metric_table(comparison_std_df, "Sample standard deviations across seeds (ddof=1)")
    for metric, winners in metric_winners(comparison_df).items():
        print(f"Best {metric}: {', '.join(winners) if winners else 'undefined for all models'}")
    delta = comparison_df.loc["QLoRA"] - comparison_df.loc["Baseline"]
    print("QLoRA minus Baseline: " + ", ".join(f"{metric} {value:+.4f}" for metric, value in delta.items()))
    print("Complete metrics, validity rates, and raw predictions saved in:", experiment_dir)

else:
    print("Result tables require completed test evaluation for every configured seed.")


Table 1: Performance Across Testing Seeds


examples  valid_examples  recovered_output_rate  \
seed model                                                       
42   Baseline       389             389                 0.0000   
     QLoRA          389             389                 0.0000   
123  Baseline       278             278                 0.0000   
     QLoRA          278             278                 0.0000   
2026 Baseline       396             396                 0.0000   
     QLoRA          396             396                 0.0000   

               invalid_output_rate  exact_match_accuracy  \
seed model                                                 
42   Baseline               0.0000                0.7712   
     QLoRA                  0.0000                0.7635   
123  Baseline               0.0000                0.6942   
     QLoRA                  0.0000                0.7626   
2026 Baseline               0.0000                0.7096   
     QLoRA                  0.0000                0.7778   

               per_criterion_accuracy  precision  recall     f1  \
seed model                                                        
42   Baseline                  0.8949     0.9356  0.8685 0.9008   
     QLoRA                     0.8883     0.8534  0.9622 0.9045   
123  Baseline                  0.8228     0.8821  0.7838 0.8301   
     QLoRA                     0.8657     0.8836  0.8716 0.8776   
2026 Baseline                  0.8510     0.9426  0.7518 0.8365   
     QLoRA                     0.8889     0.8839  0.8988 0.8913   

               final_score_exact_accuracy  final_score_within_one_accuracy  \
seed model                                                                   
42   Baseline                      0.7712                           0.9897   
     QLoRA                         0.7686                           0.9871   
123  Baseline                      0.7086                           0.9892   
     QLoRA                         0.7770                           0.9964   
2026 Baseline                      0.7172                           0.9899   
     QLoRA                         0.7854                           1.0000   

               final_score_mae  final_score_rmse  final_score_pearson  \
seed model                                                              
42   Baseline           0.2468            0.5600               0.8672   
     QLoRA              0.2519            0.5691               0.8832   
123  Baseline           0.3129            0.6204               0.8130   
     QLoRA              0.2302            0.5018               0.8755   
2026 Baseline           0.2929            0.5596               0.8290   
     QLoRA              0.2146            0.4633               0.8603   

               final_score_qwk  
seed model                      
42   Baseline           0.8606  
     QLoRA              0.8650  
123  Baseline           0.8065  
     QLoRA              0.8741  
2026 Baseline           0.8051  
     QLoRA              0.8599

Table 2: Average Performance Across Testing Seeds


,examples,valid_examples,recovered_output_rate,invalid_output_rate,exact_match_accuracy,per_criterion_accuracy,precision,recall,f1,final_score_exact_accuracy,final_score_within_one_accuracy,final_score_mae,final_score_rmse,final_score_pearson,final_score_qwk
model,,,,,,,,,,,,,,,
Baseline,354.3333,354.3333,0.0000,0.0000,0.7250,0.8562,0.9201,0.8014,0.8558,0.7323,0.9896,0.2842,0.5800,0.8364,0.8241
QLoRA,354.3333,354.3333,0.0000,0.0000,0.7680,0.8809,0.8736,0.9109,0.8911,0.7770,0.9945,0.2323,0.5114,0.8730,0.8663


Mean Qwen3.5-27B baseline versus extended QLoRA (r=48, alpha=96; +in_proj_qkv, +out_proj); highlighted cells are metric winners.


,QWK,Pearson,RMSE,MAE,Accuracy
model,,,,,
Baseline,0.8241,0.8364,0.5800,0.2842,0.7323
QLoRA,0.8663,0.8730,0.5114,0.2323,0.7770


Sample standard deviations across seeds (ddof=1)


,QWK,Pearson,RMSE,MAE,Accuracy
model,,,,,
Baseline,0.0317,0.0278,0.0350,0.0339,0.0339
QLoRA,0.0072,0.0117,0.0536,0.0187,0.0084


Best QWK: QLoRA
Best Pearson: QLoRA
Best RMSE: QLoRA
Best MAE: QLoRA
Best Accuracy: QLoRA
QLoRA minus Baseline: QWK +0.0423, Pearson +0.0366, RMSE -0.0686, MAE -0.0520, Accuracy +0.0447
Complete metrics, validity rates, and raw predictions saved in: qwen35-27b-grading-qlora-inprojqkv-outproj-multiseed/run-20261023T050357-563976Z


## Optional upload of one evaluated adapter

- Upload is disabled by default. Choose `UPLOAD_SEED` and `HF_REPO_ID` before enabling `RUN_HF_UPLOAD`.
- Require completed training and all six held-out evaluations. Test results do not automatically choose the published adapter.
- Upload the selected `seed_<seed>/alpha_<value>/` directory, including adapter, tokenizer, and training/evaluation artifacts; exclude intermediate `checkpoint-*` folders.
- `login()` requests Hugging Face authentication. The target repository must exist and the authenticated account must have write access.


In [24]:
if RUN_HF_UPLOAD:
    if not (RUN_TRAINING and RUN_TEST_EVALUATION):
        raise RuntimeError(
            "Set RUN_TRAINING=True and RUN_TEST_EVALUATION=True before uploading."
        )
    if not evaluation_completed:
        raise RuntimeError("Complete the held-out evaluation cells before uploading.")
    if UPLOAD_ALPHA not in LORA_ALPHAS:
        raise ValueError("UPLOAD_ALPHA must be one of LORA_ALPHAS.")
    if UPLOAD_SEED not in TRAINING_SEEDS:
        raise ValueError("UPLOAD_SEED must be one of TRAINING_SEEDS.")
    adapter_dir = experiment_dir / f"seed_{UPLOAD_SEED}" / f"alpha_{UPLOAD_ALPHA}"
    if not (adapter_dir / "adapter_config.json").is_file():
        raise RuntimeError(f"Selected adapter is missing: {adapter_dir}")
    login()
    upload_result = upload_folder(
        folder_path=str(adapter_dir),
        repo_id=HF_REPO_ID,
        repo_type="model",
        ignore_patterns=["checkpoint-*"],
        commit_message=f"Upload evaluated Qwen3.5-27B QLoRA +in_proj_qkv +out_proj r=48 alpha={UPLOAD_ALPHA} seed={UPLOAD_SEED} adapter",
    )
    print("Uploaded evaluated adapter to:", upload_result)
else:
    print("Hugging Face upload skipped. Set RUN_HF_UPLOAD=True to publish.")


Hugging Face upload skipped. Set RUN_HF_UPLOAD=True to publish.
